In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import requests as req
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,roc_auc_score
from datetime import datetime,timedelta
import pytz

In [44]:
API_KEY = 'f38bb78f99b35bcb38646b2b9d13be59'
BASE_URL = 'https://api.openweathermap.org/data/2.5/'

FETCH CURRENT DATA

In [45]:
def get_current_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"

    resposne = req.get(url)
    data = resposne.json()
    return{
        'city' : data['name'],
        'current_temp' : round(data['main']['temp']),
        'feels_like' : round(data['main']['feels_like']),
        'temp_min' : round(data['main']['temp_min']),
        'temp_max' : round(data['main']['temp_max']),
        'humidity' : round(data['main']['humidity']),
        'description' : data['weather'][0]['description'],
        'country' : data['sys']['country'],
        'wind_gust_dir' : data['wind']['deg'],
        'pressure' : data['main']['pressure'],
        'wind_gust_speed' : data['wind']['speed']

    }


In [46]:
def read_historical_data(filename):
    df = pd.read_csv(filename)
    df.dropna()
    df = df.drop_duplicates()

    return df

In [53]:
def prepare_data(data):
    encode = LabelEncoder()
    data['WindGustDir'] = encode.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = encode.fit_transform(data['RainTomorrow'])

    X = data[['MinTemp','MaxTemp','WindGustDir','WindGustSpeed','Humidity','Pressure','Temp']]
    y = data['RainTomorrow']


    return X,y,encode

In [54]:
def train_model(X,y):

    X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)

    model  = RandomForestClassifier(n_estimators=100,random_state=42)

    model.fit(X_train,y_train)

    y_pred = model.predict(X_test)

    print(mean_squared_error(y_test,y_pred))
    print(mean_absolute_error(y_test,y_pred))

    print(r2_score(y_test,y_pred))

    return model


In [55]:
def prepare_forecasting(data,feature):

    X,y = [], []

    for i in range(len(data) - 1):
        X.append(data[feature].iloc[i])
        y.append(data[feature].iloc[i] + 1)

    X = np.array(X).reshape(-1,1)
    y = np.array(y)

    return X,y

In [56]:
def train_regressor_model(X,y):
    model = RandomForestRegressor(n_estimators=100,random_state=42)
    model.fit(X,y)

    return model

In [57]:
def predict_future(model,current_value):

    predictions = [current_value]
    for i in range(5):
        next_value = model.predict(np.array([[predictions[-1]]]))

        predictions.append(next_value[0])
        
    return predictions[1:]


In [63]:
def weather_view():
    city = input("Enter your city name ")
    current_weather = get_current_weather(city)

    historical_data = read_historical_data('weather.csv')


    X, y, encode = prepare_data(historical_data)

    forecast_model = train_model(X,y)

    wind_deg = current_weather['wind_gust_dir'] % 360

    compass_points = [
        ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75), ("N", 348.75, 360)
    ]

    compass_direction = next(point for point, start, end in compass_points if start <= wind_deg < end)

    compass_direction_encoded = encode.transform([compass_direction])[0] if compass_direction in encode.classes_ else -1

    current_data = {
        'MinTemp': current_weather['temp_min'],
        'MaxTemp': current_weather['temp_max'],
        'WindGustDir': compass_direction_encoded,
        'WindGustSpeed' : current_weather['wind_gust_speed'],
        'Humidity' : current_weather['humidity'],
        "Pressure"  : current_weather['pressure'],
        "Temp" : current_weather['current_temp'],
    }

    current_df = pd.DataFrame([current_data])

    rainfal_prediction = forecast_model.predict(current_df)[0]

    X_temp, y_temp = prepare_forecasting(historical_data, 'Temp')
    X_humid, y_humid = prepare_forecasting(historical_data,'Humidity')

    temperature_model = train_regressor_model(X_temp,y_temp)
    humidity_model = train_regressor_model(X_humid,y_humid)

    future_temp = predict_future(temperature_model,current_weather['temp_min'])
    future_humid = predict_future(humidity_model,current_weather['humidity'])


    timezone = pytz.timezone('Asia/Kolkata')
    now = datetime.now(timezone)
    
    next_hour = now + timedelta(hours=1)
    next_hour = next_hour.replace(minute=0, second=0, microsecond=0)
    future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]



    print(f"City : {city}, {current_weather['country']}")
    print(f"Current Temperature : {current_weather['current_temp']}")
    print(f"Minimum Temperature : {current_weather['temp_min']}")
    print(f"Maximum Temperature : {current_weather['temp_max']}")
    print(f"Humidity : {current_weather['humidity']}")

    print(f"Rain Prediction : {'Yes' if rainfal_prediction else 'No'}")

    print("\n----------------- Future temperature forecast ----------------------")

    for time,temp in zip(future_times,future_temp):
        print(f"{time} : {round(temp,1)} Celcius")

    print("\n----------------- Future Humidity forecast ----------------------")

    for time,humidity in zip(future_times,future_humid):
        print(f"{time} : {round(humidity,1)} Celcius")



weather_view()


0.16216216216216217
0.16216216216216217
0.04310344827586221
City : naihati, IN
Current Temperature : 35
Minimum Temperature : 31
Maximum Temperature : 35
Humidity : 56
Rain Prediction : No

----------------- Future temperature forecast ----------------------
12:00 : 32.0 Celcius
13:00 : 33.0 Celcius
14:00 : 34.0 Celcius
15:00 : 35.0 Celcius
16:00 : 35.4 Celcius

----------------- Future Humidity forecast ----------------------
12:00 : 57.0 Celcius
13:00 : 58.0 Celcius
14:00 : 59.0 Celcius
15:00 : 59.9 Celcius
16:00 : 61.0 Celcius
